# IMPORT LIBRAIRIES

In [16]:
import os, random, math
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
import time
from tqdm.auto import tqdm
import numpy as np


# Dataset

In [ ]:
# Preparation of the data we will use for training
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff")

class RandomPatchSigmaMapDataset(Dataset):
    """
    Randomly sample patches from clean images and add Gaussian noise with random sigma.
    Returns:
      inp   (4,H,W) = noisy_rgb (3) + sigma_map (1)
      clean (3,H,W)
    """
    def __init__(self, clean_dir: str, patch: int = 30, sigma_min: float = 0.0, sigma_max: float = 50.0):
        self.paths = [
            os.path.join(clean_dir, f) for f in os.listdir(clean_dir)
            if f.lower().endswith(IMG_EXT)
        ]
        if not self.paths:
            raise ValueError(f"No images found in {clean_dir}")
        self.patch = patch
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

    def __len__(self):
        return 1000000

    def _random_crop(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w < self.patch or h < self.patch:
            img = img.resize((max(w, self.patch), max(h, self.patch)))
            w, h = img.size
        x0 = random.randint(0, w - self.patch)
        y0 = random.randint(0, h - self.patch)
        return img.crop((x0, y0, x0 + self.patch, y0 + self.patch))

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.5:
            img = TF.vflip(img)
        k = random.randint(0, 3)
        if k:
            img = img.rotate(90 * k)
        return img

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = Image.open(path).convert("RGB")
        img = self._random_crop(img)
        img = self._augment(img)

        clean = TF.to_tensor(img)

        sigma = random.uniform(self.sigma_min, self.sigma_max)
        noise = torch.randn_like(clean) * (sigma / 255.0)
        noisy = (clean + noise)

        sigma_map = torch.full((1, clean.shape[1], clean.shape[2]), sigma / 255.0)
        inp = torch.cat([noisy, sigma_map], dim=0)

        return inp, clean


# ICRNN CONDITIONNE AVEC UNE SIGMA MAP

In [ ]:
# Model: IRCNN (7-layer dilated) + sigma map
class IRCNNSigmaMap(nn.Module):
    """
    Input:  (B,4,H,W) = noisy RGB + sigma map
    Output: (B,3,H,W) clean
    Residual learning: predict noise implicitly then subtract
    """
    def __init__(self, features: int = 64):
        super().__init__()
        dilations = [1, 2, 3, 4, 3, 2, 1]
        layers = []

        d = dilations[0]
        layers += [
            nn.Conv2d(4, features, 3, padding=d, dilation=d, bias=True),
            nn.ReLU(inplace=True),
        ]

        for d in dilations[1:-1]:
            layers += [
                nn.Conv2d(features, features, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ]

        d = dilations[-1]
        layers += [nn.Conv2d(features, 3, 3, padding=d, dilation=d, bias=True)]
        self.net = nn.Sequential(*layers)

    def forward(self, inp):
        pred_noise = self.net(inp)
        noisy = inp[:, :3, :, :]
        clean = (noisy - pred_noise)
        return clean

# Train

In [19]:
# Train
def train(
    clean_dir=r"./BSDS300/images/train",
    out_dir="weights_ircnn_sigmap",
    patch=35,
    sigma_min=2.0,
    sigma_max=40.0,
    batch_size=8,
    steps_per_epoch=1000,
    max_epochs=10,
    lr0=1e-3,
    lr1=1e-4,
    plateau_epochs=5,
    log_every=50
):
    os.makedirs(out_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    ds = RandomPatchSigmaMapDataset(clean_dir, patch=patch, sigma_min=sigma_min, sigma_max=sigma_max)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,              
        pin_memory=(device == "cuda"),
        drop_last=True
    )

    model = IRCNNSigmaMap().to(device).train()
    opt = Adam(model.parameters(), lr=lr0)
    loss_fn = nn.MSELoss()
    scaler = GradScaler(enabled=(device == "cuda"))

    best = float("inf")
    stagnant = 0
    using_lr1 = False

    global_step = 0

    for epoch in range(1, max_epochs + 1):
        running = 0.0
        start_t = time.time()

        pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch}/{max_epochs}", leave=True)
        for step, (inp, clean) in enumerate(dl):
            if step >= steps_per_epoch:
                break

            inp = inp.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)

            noisy = inp[:, :3, :, :]
            target_noise = noisy - clean

            with autocast(enabled=(device == "cuda")):
                pred_clean = model(inp)
                pred_noise = noisy - pred_clean
                loss = loss_fn(pred_noise, target_noise)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running += loss.item()
            global_step += 1

            # logs
            if (step + 1) % log_every == 0:
                avg_so_far = running / (step + 1)
                elapsed = time.time() - start_t
                it_s = (step + 1) / max(elapsed, 1e-9)
                pbar.set_postfix({
                    "loss": f"{avg_so_far:.5f}",
                    "lr": f"{opt.param_groups[0]['lr']:.1e}",
                    "it/s": f"{it_s:.2f}"
                })

            pbar.update(1)

        pbar.close()

        avg = running / max(1, steps_per_epoch)
        ckpt = os.path.join(out_dir, f"ircnn_sigmap_epoch{epoch:02d}.pth")
        torch.save({"model": model.state_dict(), "epoch": epoch}, ckpt)

        print(f"Epoch {epoch:02d} done | avg_loss={avg:.6f} | lr={opt.param_groups[0]['lr']:.1e}")

        # LR schedule + early stop like before
        if avg < best - 1e-7:
            best = avg
            stagnant = 0
        else:
            stagnant += 1

        if (not using_lr1) and stagnant >= plateau_epochs:
            for g in opt.param_groups:
                g["lr"] = lr1
            using_lr1 = True
            stagnant = 0
            print(f"Switch LR to {lr1}")

        if using_lr1 and stagnant >= plateau_epochs:
            print("Early stop: loss plateaued.")
            break

    final_path = os.path.join(out_dir, "ircnn_sigmap_final.pth")
    torch.save({"model": model.state_dict()}, final_path)
    print(f"Saved: {final_path}")


train()

device: cuda


C:\Users\barra\AppData\Local\Temp\ipykernel_17936\3167697430.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))


Epoch 1/10:   0%|          | 0/1000 [00:00<?, ?it/s]

C:\Users\barra\AppData\Local\Temp\ipykernel_17936\3167697430.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):


KeyboardInterrupt: 